<a href="https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

print("Token loaded:", token is not None)
print("Token length:", len(token) if token else 0)

api = HfApi(token=token)
me = api.whoami()

print("Hugging Face login:", me["name"])

Token loaded: True
Token length: 37
Hugging Face login: Fbeva


In [5]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB → Hugging Face connection ready.")

DuckDB → Hugging Face connection ready.


In [6]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one content item for one client on one report date. The analysis uses March 2026 as the development window.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            CAST(report_date AS VARCHAR)
            || '|' || CAST(client_hash_id AS VARCHAR)
            || '|' || CAST(content_hash_id AS VARCHAR)
        ) AS unique_row_keys
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

grain_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────┐
│ total_rows │ unique_row_keys │
│   int64    │      int64      │
├────────────┼─────────────────┤
│    9841378 │         9841378 │
└────────────┴─────────────────┘



## 2. Fields: feature / label / context / excluded

## Sort every field you plan to touch into these four buckets. Excluded needs a why.

Features: gsc_impressions, gsc_clicks, gsc_sum_position.

Label/proxy: Current engagement opportunity based on multiple observed engagement signals.

Context: report_date, client_hash_id, content_hash_id, month.

Excluded: Data-availability flags (client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available) are used to check whether data is available, not as engagement signals. GA4 and session-source fields are excluded from the core feature set because they are unavailable for a substantial portion of the March slice. gsc_avg_position is excluded because it has substantial missingness in this slice.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields = con.sql("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

fields.show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Grain check

In [10]:
grain_check = con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT
            CAST(report_date AS VARCHAR)
            || '|' || CAST(client_hash_id AS VARCHAR)
            || '|' || CAST(content_hash_id AS VARCHAR)
        ) AS unique_row_keys
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

grain_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────┐
│ total_rows │ unique_row_keys │
│   int64    │      int64      │
├────────────┼─────────────────┤
│    9841378 │         9841378 │
└────────────┴─────────────────┘



## Formal row count + date window
Verification Query 2

In [11]:
march_summary = con.sql("""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

march_summary.show()

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



#Availability check:
March 2026, GSC data is available for 3,611,061 rows and GA4 data is available for 413,966 rows. The client-level flags are higher, showing that having a data source does not mean data is available for every row

In [12]:
availability_check = con.sql("""
    SELECT 'Client has GSC' AS check_name,
           COUNT(*) FILTER (WHERE client_has_gsc IS TRUE) AS rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    UNION ALL

    SELECT 'GSC data available',
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    UNION ALL

    SELECT 'Client has GA4',
           COUNT(*) FILTER (WHERE client_has_ga4 IS TRUE)
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )

    UNION ALL

    SELECT 'GA4 data available',
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""")

availability_check.show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬─────────┐
│     check_name     │  rows   │
│      varchar       │  int64  │
├────────────────────┼─────────┤
│ Client has GSC     │ 9841378 │
│ GSC data available │ 3611061 │
│ Client has GA4     │ 6822637 │
│ GA4 data available │  413966 │
└────────────────────┴─────────┘



In [13]:
con.sql("""
    SELECT
        column_name,
        column_type
    FROM (
        DESCRIBE
        SELECT *
        FROM read_parquet(
            'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        )
    )
""").show()

┌────────────────────┬─────────────┐
│    column_name     │ column_type │
│      varchar       │   varchar   │
├────────────────────┼─────────────┤
│ report_date        │ DATE        │
│ client_hash_id     │ VARCHAR     │
│ content_hash_id    │ VARCHAR     │
│ client_has_gsc     │ BOOLEAN     │
│ client_has_ga4     │ BOOLEAN     │
│ gsc_data_available │ BOOLEAN     │
│ ga4_data_available │ BOOLEAN     │
│ gsc_impressions    │ BIGINT      │
│ gsc_clicks         │ BIGINT      │
│ gsc_sum_position   │ BIGINT      │
│      ·             │   ·         │
│      ·             │   ·         │
│      ·             │   ·         │
│ sessions_ai        │ BIGINT      │
│ ai_chatgpt         │ BIGINT      │
│ ai_perplexity      │ BIGINT      │
│ ai_gemini          │ BIGINT      │
│ ai_copilot         │ BIGINT      │
│ ai_claude          │ BIGINT      │
│ ai_meta            │ BIGINT      │
│ ai_other           │ BIGINT      │
│ scroll_events      │ BIGINT      │
│ month              │ VARCHAR     │
├

# Missing Values

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_gsc_clicks,
        COUNT(*) FILTER (WHERE gsc_sum_position IS NULL) AS missing_gsc_sum_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────────┬────────────────────┬──────────────────────────┐
│ total_rows │ missing_gsc_impressions │ missing_gsc_clicks │ missing_gsc_sum_position │
│   int64    │          int64          │       int64        │          int64           │
├────────────┼─────────────────────────┼────────────────────┼──────────────────────────┤
│    9841378 │                       0 │                  0 │                        0 │
└────────────┴─────────────────────────┴────────────────────┴──────────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits

Unbalanced history: Not every page has the same amount or continuity of historical data, so comparisons of longer-term performance or trends may not be equally reliable across pages.
Uneven source availability: GSC and GA4 data are not available for all rows in the same way. Therefore, engagement signals from different sources cannot always be compared across the full slice.
Time-window overlap: Features and outcome windows must be kept separate when constructing an engagement-opportunity proxy. Overlapping windows could allow information from the outcome period to influence the features and create leakage.

Named limitation: The main limitation of this slice is uneven source availability, meaning that some engagement signals are unavailable for a substantial portion of the rows.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check source availability in the March 2026 slice

con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").show()

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.